# Tick Extract Raman Analysis

Run this notebook to generate the figure in the pub.

In [ ]:
from pathlib import Path

import arcadia_pycolor as apc
import ramanspy as rp
from plotly.subplots import make_subplots

from raman_analysis.utils import collect_spectra_from_dir, plot_spectra_in_dir

%load_ext autoreload
%autoreload 2

apc.plotly.setup()

parent_dir = Path("../data")

fig_h, fig_w = 300, 1000
spectrum_lims = (700, 3200)
y_offset = (1000, 5000)

colors = ["gray", "red"]

# Processing pipeline for Raman spectra
pipeline = rp.preprocessing.Pipeline(
    [
        rp.preprocessing.misc.Cropper(region=spectrum_lims),
        rp.preprocessing.despike.WhitakerHayes(),
        rp.preprocessing.denoise.SavGol(window_length=5, polyorder=3),
        rp.preprocessing.baseline.ModPoly(poly_order=5),
    ]
)

### Wasatch 532 on 2025-07-24

In [ ]:
data_dir_sers = parent_dir / "sers-on-fraction"

data_list_sers, sample_df_sers = collect_spectra_from_dir(data_dir_sers, "wasatch")

f, _, _ = plot_spectra_in_dir(
    data_list_sers,
    col_to_plot="Intensity",
    data_dir=data_dir_sers,
    rp_pipeline=pipeline,
    colors=colors,
    y_offset=y_offset[1],
)


data_dir_whole = parent_dir / "whole-tick-extract"

data_list_whole, sample_df_whole = collect_spectra_from_dir(data_dir_whole, "wasatch")

f2, _, _ = plot_spectra_in_dir(
    data_list_whole,
    col_to_plot="Intensity",
    data_dir=data_dir_whole,
    rp_pipeline=pipeline,
    colors=colors,
    y_offset=y_offset[0],
)

# Combine two figures into one with two subplots
combined_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=["Raman spectra of whole tick extract", "SERS of fractionated tick extract"],
    vertical_spacing=0.15,
)

for trace in f.data:
    combined_fig.add_trace(trace, row=2, col=1)

for trace in f2.data:
    combined_fig.add_trace(trace, row=1, col=1)


combined_fig.update_layout(height=fig_h * 2, width=fig_w, showlegend=True)
combined_fig.update_xaxes(title_text=f2.layout.xaxis.title.text, row=2, col=1)
combined_fig.update_yaxes(title_text="Intensity (a.u.)", row=1, col=1)
combined_fig.update_yaxes(title_text="Intensity (a.u.)", row=2, col=1)

combined_fig.update_layout(template="plotly_white")

apc.plotly.style_plot(combined_fig, monospaced_axes="all")

combined_fig.show()